In [4]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

In [6]:
pip install selenium webdriver-manager

  Using cached certifi-2026.2.25-py3-none-any.whl.metadata (2.5 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached urllib3-2.6.3-py3-none-any.whl.metadata (6.9 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 8.0 MB/s eta 0:00:00a 0:00:01
Using cached certifi-2026.2.25-py3-none-any.whl (153 kB)
Using cached typing_extensions-4.15.0-py3-none-any.whl (44 kB)
Using cached urllib3-2.6.3-py3-none-any.whl (131 kB)
Using cached h11-0.16.0-py3-none-any.whl (37 kB)
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.2.3
    Uninstalling urllib3-2.2.3:
      Successfully uninstalled urllib3-2.2.3
  Attempting uninstall: typing_extensions
    Found existing installation: typing_extensions 4.11.0
    Uninstalling typing_extensions-4.11.0:
      Successfully uninstalled typing_extensions-4.11.0
  Attempting uninstall: h11
    Found existing installation: h11 0.14.

In [105]:
import time
import random
import csv
import logging
import concurrent.futures
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

In [68]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logging.getLogger('WDM').setLevel(logging.WARNING)

In [69]:
def get_driver():
    options = Options()
    options.add_argument('--headless')
    options.add_argument('--disable-gpu')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--window-size=1920,1080')
    options.add_argument('--remote-debugging-port=0')
    options.page_load_strategy = 'eager'
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

In [95]:
def get_reviews(url):
    time.sleep(0.5)
    driver = None
    try:
        driver = get_driver()
        driver.set_script_timeout(15)
        driver.set_page_load_timeout(7)
        driver.get(url)
        WebDriverWait(driver, 5).until(EC.visibility_of_element_located((By.CLASS_NAME, 'review-list')))
        more_buttons = driver.find_elements(By.XPATH, '//div[contains(@class, "comment")]//a[contains(text(), "More")]')
        for btn in more_buttons:
            driver.execute_script('arguments[0].click();', btn)
        comments = driver.find_elements(By.CLASS_NAME, 'comment')
        data = 'ЮРА'.join([comment.text.replace('\n', ' ').strip() for comment in comments])
        logging.info(f'Отзывы нашлись с кайфом: {url}')
        return {'url': url, 'status': 'Success', 'reviews': data}
    except Exception as e:
        logging.error(f'Биля, упало {url}: {str(e)[:50]}')
        return {'url': url, 'status': 'Failed', 'reviews': ''}
    finally:
        if driver: driver.quit()

Тут смотрел про многопоточность:
https://docs.python.org/3/library/concurrent.futures.html

In [102]:
def parse(urls):
    start = time.time()
    with open("results15.csv", "w", newline="", encoding="utf-8") as file:
        counter = 0
        writer = csv.writer(file)
        writer.writerow(['url','status','reviews'])
        with concurrent.futures.ThreadPoolExecutor(max_workers=15) as executor:
            results = {executor.submit(get_reviews, url): url for url in urls}
            for result in concurrent.futures.as_completed(results):
                res = result.result()
                writer.writerow([res['url'],res['status'],res['reviews']])
                counter+=1
                logging.info(f'ПРОГРЕССИРУЮ НА {counter} ССЫЛОК ИЗ {len(urls)}')
                if counter == len(urls)//2:
                    logging.info('Емае, еще столько жееее')
    logging.info(f'THE END! {len(school_urls)} Ссылок за {round((time.time()-start)/60,2)} минут')

In [103]:
reviews = parse(school_urls[:10])

2026-04-10 03:58:37,364 - INFO - Отзывы нашлись с кайфом: https://www.greatschools.org/florida/north-port/2800-Glenallen-Elementary-School/reviews
2026-04-10 03:58:37,368 - INFO - Отзывы нашлись с кайфом: https://www.greatschools.org/florida/north-port/2800-Glenallen-Elementary-School/reviews
2026-04-10 03:58:37,508 - INFO - ПРОГРЕССИРУЮ НА 1 ССЫЛОК ИЗ 10
2026-04-10 03:58:37,509 - INFO - ПРОГРЕССИРУЮ НА 2 ССЫЛОК ИЗ 10
2026-04-10 03:58:37,539 - INFO - Отзывы нашлись с кайфом: https://www.greatschools.org/florida/north-port/7848-Lamarque-Elementary-School/reviews
2026-04-10 03:58:37,548 - INFO - Отзывы нашлись с кайфом: https://www.greatschools.org/florida/north-port/5294-Cranberry-Elementary-School/reviews
2026-04-10 03:58:37,571 - INFO - Отзывы нашлись с кайфом: https://www.greatschools.org/florida/north-port/7848-Lamarque-Elementary-School/reviews
2026-04-10 03:58:37,644 - INFO - ПРОГРЕССИРУЮ НА 3 ССЫЛОК ИЗ 10
2026-04-10 03:58:37,655 - INFO - ПРОГРЕССИРУЮ НА 4 ССЫЛОК ИЗ 10
2026-04-10 